# Соколов / не Соколов: ResNet18 и ArcFace

Полный пользовательский запуск на Kaggle GPU. В Settings включите **GPU** и **Internet**.
Инструкция: `docs/KAGGLE.md`. Измеренные результаты появляются только после обучения.

Разбиение случайное 70/15/15: лекции пересекаются между выборками, поэтому test не является проверкой на новой лекции.

Подключите официальный **PyTorch iResNet50/MS1MV3 checkpoint** через Add Input (приватный Dataset). Это файл `backbone.pth`, а не ONNX и не torchvision ResNet50.
Код можно загрузить как Dataset с `project_bundle.zip`; это позволяет запускать локальные изменения до публикации кода в GitHub.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile
WORK = Path('/kaggle/working')
REPO = WORK / 'Sokolov-s-Classifier'
PROJECT_REF = 'main'
archives = list(Path('/kaggle/input').rglob('project_bundle.zip'))
if not REPO.exists():
    if len(archives) == 1:
        REPO.mkdir(parents=True)
        with zipfile.ZipFile(archives[0]) as archive:
            assert all((REPO / name).resolve().is_relative_to(REPO.resolve()) for name in archive.namelist())
            archive.extractall(REPO)
    elif len(archives) > 1:
        raise ValueError('Подключите только один project_bundle.zip')
    else:
        subprocess.run(['git', 'clone', '--branch', PROJECT_REF,
                        'https://github.com/nikitiva0/Sokolov-s-Classifier.git', str(REPO)], check=True)
os.chdir(REPO)
if not (REPO / 'requirements-kaggle.txt').exists():
    raise RuntimeError('На GitHub ещё старый каркас. Подключите project_bundle.zip или опубликуйте новый код.')
os.environ['TORCH_HOME'] = str(WORK / 'cache/torch')
os.environ['KAGGLEHUB_CACHE'] = str(WORK / 'cache/kagglehub')
subprocess.run(['nvidia-smi'], check=True)


## Установка

После первой установки при запросе Kaggle перезапустите kernel и выполните ячейки заново. Python 3.11/3.12; GPU-совместимость проверяется ниже. Код проекта не меняет системную CUDA.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)


## Параметры

Укажите точный путь к официальному checkpoint в `/kaggle/input/.../backbone.pth`. Если таких файлов несколько, автоматический выбор запрещён. SHA-256 можно задать для проверки; для файла, подключённого вручную, фактический хеш в любом случае сохраняется.

`FAIRFACE_DIR = ''` означает автоматическую загрузку. Для уже подключённого FairFace укажите папку с изображениями. Весь набор скачивается/подключается один раз, затем детерминированно выбираются 5000 негативов.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Включите GPU в настройках Kaggle'
print(torch.__version__, torch.cuda.get_device_name(0))
ARCFACE_CHECKPOINT = ''  # например /kaggle/input/my-arcface/backbone.pth
ARCFACE_SHA256 = ''
FAIRFACE_DIR = ''        # необязательный /kaggle/input/... с FairFace
RUN_ID = 'fairface5000_seed42_v1'
RUN_DIR = WORK / 'runs' / RUN_ID
RESUME = False
if not ARCFACE_CHECKPOINT:
    candidates = list(Path('/kaggle/input').rglob('backbone.pth'))
    if len(candidates) != 1:
        raise ValueError('Укажите ARCFACE_CHECKPOINT: нужен ровно один официальный backbone.pth')
    ARCFACE_CHECKPOINT = str(candidates[0])
extra = ['--arcface-checkpoint', ARCFACE_CHECKPOINT]
if ARCFACE_SHA256:
    extra += ['--arcface-sha256', ARCFACE_SHA256]
if FAIRFACE_DIR:
    extra += ['--fairface-dir', FAIRFACE_DIR]
def cli(*arguments):
    subprocess.run([sys.executable, '-m', 'sokolov', *map(str, arguments)], check=True)
cli('check-config')
cli('check-weights', *extra)


## Загрузка, аудит и обучение

Следующая команда запускает весь конвейер: загрузку → проверку дублей → общий split → SCRFD и выравнивание лиц → две модели → test → выбор победителя по validation → генерацию из пяти шумов → сравнительный отчёт. Для повторного запуска того же каталога используйте `RESUME=True`; изменение данных или параметров требует нового каталога. Test не участвует в выборе моделей и порогов.

In [ ]:
arguments = ['run-all', '--run-dir', str(RUN_DIR), *extra]
if RESUME:
    arguments += ['--resume']
cli(*arguments)


## Результаты

Файлы отчёта содержат реальные измерения этого запуска. Checkpoints и полные предсказания остаются в `/kaggle/working/runs/...`. Сохраните версию ноутбука через Save Version / Run All, чтобы Kaggle сохранил выходные файлы.

In [ ]:
from IPython.display import HTML, display, FileLink
import shutil
report = RUN_DIR / 'report'
display(HTML((report / 'report.html').read_text()))
report_zip = shutil.make_archive(str(WORK / (RUN_ID + '_report')), 'zip', report)
display(FileLink(report_zip))
print('Победитель для генерации:', RUN_DIR / 'best_model.pt')
print('Модели:', RUN_DIR / 'resnet18/best.pt', RUN_DIR / 'arcface/best.pt')


## Повторная генерация из лучшей сохранённой модели

Общий запуск уже создал пять вариантов и включил их в отчёт. Эта ячейка нужна, если обучение было выполнено раньше или вы хотите отдельно восстановить генерацию из `best_model.pt`. Данные и исходные pretrained веса для этого не нужны.

Повторный запуск с теми же настройками использует проверенные результаты. Для других ограничений/seed используйте `--generation-config ... --output-dir ...` с новой папкой. Отчёт автоматически использует основной каталог `synthesis/`.


In [ ]:
REGENERATE = False
if REGENERATE:
    cli('synthesize', '--run-dir', RUN_DIR)
    cli('report', '--run-dir', RUN_DIR)
    display(HTML((RUN_DIR / 'report/report.html').read_text()))


## Необязательно: отдельная публикация в GitHub

После просмотра отчёта добавьте Kaggle Secret **GITHUB_TOKEN** с доступом Contents: write только к целевому репозиторию. Эта ячейка выполняется вручную: публикует отчёт и графики в `reports/<RUN_ID>/` ветки main. Веса, датасеты и токен не включаются. Убедитесь, что RUN_ID уникален.

In [ ]:
PUBLISH = False  # установите True только для отправки просмотренного отчёта
if PUBLISH:
    from kaggle_secrets import UserSecretsClient
    from sokolov.report import publish_report
    os.environ['SOKOLOV_GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
    try:
        print('Published commit:', publish_report(RUN_DIR))
    finally:
        os.environ.pop('SOKOLOV_GITHUB_TOKEN', None)
